In [1]:
import pandas as pd
import numpy as np
import os
import pickle
%pwd
os.chdir('../')
%pwd

'/media/om/volume2/MLOPS/The Ultimate MLOPS Course/youtube_project/MLOPS_youtube_CTA_Project'

In [2]:
df= pd.read_csv("./data/raw/youtube_10000_videos.csv")
df.head()

,category,channel_id,channel_name,subscriber_count,channel_view_count,channel_video_count,video_id,video_title,published_at,duration_seconds,view_count,like_count,comment_count,description,video_url
0,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,MOlaKBJ1nDA,प्रश्नावली 5.2 Class 10 Maths | NCERT Class 10...,2026-07-18T01:52:16Z,3988.0,427,21,3,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=MOlaKBJ1nDA
1,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,uXRH_uqCOXI,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-19T02:22:39Z,3606.0,528,32,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=uXRH_uqCOXI
2,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,q4Q4dg6JDE0,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-22T01:57:46Z,3985.0,520,23,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=q4Q4dg6JDE0
3,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,UTrN2vvjau8,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-23T01:56:33Z,4655.0,566,27,0,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=UTrN2vvjau8
4,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,22S8ptzHl-g,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-24T02:38:43Z,2185.0,355,23,3,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=22S8ptzHl-g


In [3]:
all_columns=df.columns
all_columns

Index(['category', 'channel_id', 'channel_name', 'subscriber_count',
       'channel_view_count', 'channel_video_count', 'video_id', 'video_title',
       'published_at', 'duration_seconds', 'view_count', 'like_count',
       'comment_count', 'description', 'video_url'],
      dtype='object')

In [4]:
required_columns=['category','subscriber_count','channel_view_count','duration_seconds', 'view_count']
df1=df[required_columns]
df1.head()

,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355


In [5]:
def remove_null(df:pd.DataFrame)->pd.DataFrame:
    df1=df.copy()
    df1.dropna(inplace=True)
    print("Total null values removed: ",(len(df)-len(df1)))
    print("percent of null values removed: ",((len(df)-len(df1))*100/len(df)))
    return df1

df2= remove_null(df1)
df2.head()

Total null values removed:  6
percent of null values removed:  0.06027122049221497


,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor)
from sklearn.linear_model import Ridge
import numpy as np

numeric_features = [
    'subscriber_count','channel_view_count','duration_seconds'
]
categorical_features = ["category"]

def combined_transform(numerical_columns:list[str],categorical_columns: list[str])->Pipeline:
    """ This finction provides the combined pipeline for numerical and  categorical features"""

    # numeric pipeline: imputer-> log transform-> standardScaler
    numeric_pipeline= Pipeline(
        steps=[
            ('imputer',SimpleImputer(strategy='median')),
            ('log_transform',FunctionTransformer(np.log1p)),
            ('scaling',StandardScaler())
        ]
    )

    # categorical_pipeline: imputer -> onehot encoding
    categorical_pipeline= Pipeline(
        steps=[
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',OneHotEncoder(handle_unknown='ignore')),
        ]
    )

    # applying above pipelines to given columns
    combined_processor= ColumnTransformer(
        transformers=[
            ('numeric',numeric_pipeline,numeric_features),
            ('categorical',categorical_pipeline,categorical_features)
        ]
    )

    return combined_processor

combined_processor_pipeline= combined_transform(numeric_features,categorical_features)

In [8]:
def model_with_combined_processor(combined_processor:Pipeline, model)->Pipeline:
    # model pipeline: processor -> model
    model_pipeline= Pipeline(
        steps=[
            ('combined_transform',combined_processor),
            ('model',model)
        ]
    )
    return model_pipeline

In [9]:
model_pipeline= model_with_combined_processor(combined_processor_pipeline,RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1))
model_pipeline

,steps,"[('combined_transform', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
X=df2.drop(columns=["view_count"])
X

,category,subscriber_count,channel_view_count,duration_seconds
0,Education,167000,5169057,3988.0
1,Education,167000,5169057,3606.0
2,Education,167000,5169057,3985.0
3,Education,167000,5169057,4655.0
4,Education,167000,5169057,2185.0
...,...,...,...,...
9950,Travel,127000,18680974,3719.0
9951,Travel,127000,18680974,4181.0
9952,Travel,127000,18680974,3743.0
9953,Travel,127000,18680974,22.0


In [11]:
y= np.log1p(df2["view_count"])
y

0        6.059123
1        6.270988
2        6.255750
3        6.340359
4        5.874931
          ...    
9950    11.109713
9951    12.354613
9952    11.358106
9953    11.902248
9954    12.598442
Name: view_count, Length: 9949, dtype: float64

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)
predictions

array([10.10213458,  9.04722738,  9.93446454, ..., 10.82662669,
       12.90376124,  7.91859683], shape=(1990,))

In [14]:
from sklearn.metrics import r2_score, mean_squared_error
y_pred= model_pipeline.predict(X_test)
r2= r2_score(y_test,y_pred)
r2

0.7814785376886162

## imports

In [15]:
import numpy as np
import pandas as pd
import yaml

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn


/home/om/Desktop/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# models dictionary

In [16]:
models= {
    "LinearRegression": LinearRegression(),
    "Ridge":Ridge(),
    "RandomForestRegressor":RandomForestRegressor(random_state=42,n_jobs=-1),
    "GradientBoostingRegressor":GradientBoostingRegressor(random_state=42),
    "XGBRegressor":XGBRegressor(random_state=42,n_jobs=-1,objective="req:squarederror")
}

# params dictionary

In [17]:
# using 'model' before the parameter for pipeline
param_grids= {
    "LinearRegression":{"model_fit_intercept":[True,False]},
    "Ridge":{"model_alpha":[0.01,0.1,1,10,100],
             "model_fit_intercept":[True,False]},
    "RandomForestRegressor":{
        "n_estimators":[100,200],
        "max_depth":[None,10,20],
        "min_samples_split":[2,5]
    },
    "GradientBoostingRegressor":{
        "model_n_estimators":[100,200],
        "model_learning_rate":[0.001,0.01,0.1],
        "model_max_depth":[3,5]
    },
    "XGBRegressor":{
        "model_n_estimators":[100,200],
        "model_learning_rate":[0.001,0.01,0.1],
        "model_max_depth":[3,5]
    }
}

# model__alpha
# The first model refers to the Pipeline step:
# ("model", Ridge())
# and alpha is the model's parameter.


In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import mlflow
import dagshub
models= {
    "LinearRegression": LinearRegression(),
    "Ridge":Ridge(),
    "RandomForestRegressor":RandomForestRegressor(random_state=42,n_jobs=-1),
    "GradientBoostingRegressor":GradientBoostingRegressor(random_state=42),
    "XGBRegressor":XGBRegressor(random_state=42,n_jobs=-1,objective="reg:squarederror")
}
# using 'model' before the parameter for pipeline
param_grids= {
    "LinearRegression":{"model__fit_intercept":[True,False]},
    "Ridge":{"model__alpha":[0.01,0.1,1,10,100],
             "model__fit_intercept":[True,False]},
    "RandomForestRegressor":{
        "model__n_estimators":[100,200],
        "model__max_depth":[None,10,20],
        "model__min_samples_split":[2,5]
    },
    "GradientBoostingRegressor":{
        "model__n_estimators":[100,200],
        "model__learning_rate":[0.001,0.01,0.1],
        "model__max_depth":[3,5]
    },
    "XGBRegressor":{
        "model__n_estimators":[100,200],
        "model__learning_rate":[0.001,0.01,0.1],
        "model__max_depth":[3,5]
    }
}

results = []
dagshub.init(repo_owner='AIforeverything', repo_name='MLOPS_youtube_CTA_Project', mlflow=True)

with mlflow.start_run(run_name="Regression_Models") as parent_run:

    for model_name, model in models.items():

        print(f"\n{'=' * 60}")
        print(f"Training: {model_name}")
        print(f"{'=' * 60}")

        # --------------------------------------------------
        # Create pipeline
        # --------------------------------------------------
        numeric_features = [
            'subscriber_count','channel_view_count','duration_seconds'
        ]
        categorical_features = ["category"]

        # numeric pipeline: imputer-> log transform-> standardScaler
        numeric_pipeline= Pipeline(
            steps=[
                ('imputer',SimpleImputer(strategy='median')),
                ('log_transform',FunctionTransformer(np.log1p)),
                ('scaling',StandardScaler())
            ]
        )
    
        # categorical_pipeline: imputer -> onehot encoding
        categorical_pipeline= Pipeline(
            steps=[
                ('imputer',SimpleImputer(strategy='most_frequent')),
                ('onehot',OneHotEncoder(handle_unknown='ignore')),
            ]
        )
    
        # applying above pipelines to given columns
        combined_processor= ColumnTransformer(
            transformers=[
                ('numeric',numeric_pipeline,numeric_features),
                ('categorical',categorical_pipeline,categorical_features)
            ]
        )
        model_pipeline= Pipeline(
                steps=[
                    ('combined_transform',combined_processor),
                    ('model',model)
                ]
            )

        # --------------------------------------------------
        # Create GridSearchCV
        # --------------------------------------------------

        grid_search = GridSearchCV(
            estimator=model_pipeline,
            param_grid=param_grids[model_name],
            cv=5,
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            return_train_score=True
        )

        # --------------------------------------------------
        # Start MLflow run for this algorithm
        # --------------------------------------------------

        with mlflow.start_run(
            run_name=model_name,
            nested=True
        ):

            # Log model name
            mlflow.log_param(
                "algorithm",
                model_name
            )

            mlflow.log_param(
                "cv_folds",
                5
            )

            mlflow.log_param(
                "scoring",
                "neg_mean_squared_error"
            )

            # --------------------------------------------------
            # Train GridSearchCV
            # --------------------------------------------------

            grid_search.fit(
                X_train,
                y_train
            )

            # --------------------------------------------------
            # Best model
            # --------------------------------------------------

            best_model = grid_search.best_estimator_

            best_params = grid_search.best_params_

            best_cv_score = grid_search.best_score_

            # --------------------------------------------------
            # Log best parameters
            # --------------------------------------------------

            mlflow.log_params(best_params)

            mlflow.log_metric(
                "best_cv_mse",
                -best_cv_score
            )

            # --------------------------------------------------
            # Test prediction
            # --------------------------------------------------

            y_pred = best_model.predict(X_test)

            # --------------------------------------------------
            # Test metrics
            # --------------------------------------------------

            mse = mean_squared_error(
                y_test,
                y_pred
            )

            rmse = np.sqrt(mse)

            mae = mean_absolute_error(
                y_test,
                y_pred
            )

            r2 = r2_score(
                y_test,
                y_pred
            )

            # --------------------------------------------------
            # Log test metrics
            # --------------------------------------------------

            mlflow.log_metric(
                "test_mse",
                mse
            )

            mlflow.log_metric(
                "test_rmse",
                rmse
            )

            mlflow.log_metric(
                "test_mae",
                mae
            )

            mlflow.log_metric(
                "test_r2",
                r2
            )

            # --------------------------------------------------
            # Log best model
            # --------------------------------------------------

            mlflow.sklearn.log_model(
                best_model,
                name="model",
                skops_trusted_types=["numpy.dtype",'xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor']
            )

            # --------------------------------------------------
            # Store result
            # --------------------------------------------------

            results.append({

                "model": model_name,

                "best_params": best_params,

                "cv_mse": -best_cv_score,

                "test_mse": mse,

                "test_rmse": rmse,

                "test_mae": mae,

                "test_r2": r2
            })

            print("Best Parameters:")
            print(best_params)

            print(f"CV MSE  : {-best_cv_score:.4f}")
            print(f"Test MSE: {mse:.4f}")
            print(f"Test RMSE: {rmse:.4f}")
            print(f"Test MAE : {mae:.4f}")
            print(f"Test R²  : {r2:.4f}")


Accessing as AIforeverything

Initialized MLflow to track repo "AIforeverything/MLOPS_youtube_CTA_Project"

Repository AIforeverything/MLOPS_youtube_CTA_Project initialized!


Training: LinearRegression
Best Parameters:
{'model__fit_intercept': False}
CV MSE  : 4.0184
Test MSE: 3.9010
Test RMSE: 1.9751
Test MAE : 1.5295
Test R²  : 0.4048
🏃 View run LinearRegression at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0/runs/8aa2b57fa4a4437e97fc66014569f04c
🧪 View experiment at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0

Training: Ridge
Best Parameters:
{'model__alpha': 1, 'model__fit_intercept': True}
CV MSE  : 4.0184
Test MSE: 3.9011
Test RMSE: 1.9751
Test MAE : 1.5296
Test R²  : 0.4048
🏃 View run Ridge at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0/runs/4861b2d08e7a427392102af5af9ba8a4
🧪 View experiment at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0

Training: RandomForestRegressor
Best Parameters:
{'model__max_depth': 20, 'model__min_samples_split': 5, 'model__n_estimators': 200}
CV MSE  : 1.4685

In [19]:
def save_model(model_save_path:str,model_pipeline:pickle)->None:
    """
    Function to load the model.
    parameters:
    inputs:
    model_save_path:str
    outputs:
    None
    """
    try: 
        with open(model_save_path,'wb') as model_dump:
            pickle.dump(model_pipeline,model_dump)
    except Exception as e:
        raise f"Error: {e}"        

# save_model("./src/model/model.pkl")

In [20]:
def load_model(model_path:str):
    """
    Function to load the model.
    parameters:
    inputs:
    model_path:str
    outputs:
    model.pkl
    """
    try:
        with open(model_path,'rb') as f:
            model= pickle.load(f)
        return model
    except Exception as e:
        raise f"Error: {e}" 

# model= load_model("./src/model/model.pkl")
# model

In [21]:
# pred_df= X_test.sample(2)
# pred_df

In [22]:
# out= model.predict(pred_df)
# out

In [23]:
# df2.iloc[pred_df.index[0],:]